### 0. 실습에 필요한 라이브러리 설치

In [ ]:
!pip install opencv-python requests numpy

In [ ]:
!mkdir images

### 1. Superb Endpoint 를 이용한 추론

In [ ]:
import requests
from requests.auth import HTTPBasicAuth

In [ ]:
URL = "https://suite-endpoint-api-apne2.superb-ai.com/endpoints/c52f53af-4a69-4e41-a20c-a598d6d99208/inference"
ACCESS_KEY = "tfexJfxqIu1n0pCVW8B7O6QTerg1GUTi2LdlaPAG"

In [ ]:
IMAGE_FILE_PATH = "100162_1.jpg"

In [ ]:
image = open(IMAGE_FILE_PATH, "rb").read()

In [ ]:
response = requests.post(
    url=URL,
    auth=HTTPBasicAuth("cheil-jedang", ACCESS_KEY),
    headers={"Content-Type": "image/jpeg"},
    data=image,
)

In [ ]:
print(response.json())

{'objects': [{'class': '오뚜기_진라면매운맛', 'score': 0.6051000952720642, 'box': [2815, 1381, 3014, 1519]}, {'class': '오뚜기_진라면매운맛', 'score': 0.6008782982826233, 'box': [3028, 781, 3348, 907]}, {'class': '오뚜기_진라면매운맛', 'score': 0.53268963098526, 'box': [2779, 1677, 2957, 1806]}, {'class': '오뚜기_진라면매운맛', 'score': 0.5283023118972778, 'box': [2680, 1379, 3012, 1522]}, {'class': '오뚜기_진라면매운맛', 'score': 0.5021231174468994, 'box': [2950, 1697, 3139, 1824]}, {'class': '오뚜기_진라면매운맛', 'score': 0.49510374665260315, 'box': [2700, 1674, 2956, 1799]}, {'class': '오뚜기_진라면매운맛', 'score': 0.4945889115333557, 'box': [2952, 1697, 3270, 1835]}, {'class': '오뚜기_진라면매운맛', 'score': 0.49237021803855896, 'box': [2646, 959, 2996, 1100]}, {'class': '오뚜기_진라면매운맛', 'score': 0.4743853807449341, 'box': [3049, 645, 3375, 791]}, {'class': '오뚜기_진라면매운맛', 'score': 0.47289687395095825, 'box': [2991, 1094, 3178, 1222]}, {'class': '오뚜기_진라면매운맛', 'score': 0.4566204845905304, 'box': [2670, 1380, 2842, 1510]}, {'class': '농심_신라면', 'score': 0.435

### 2. 결과 확인

In [ ]:
results = response.json()

In [ ]:
results.keys()

dict_keys(['objects'])

In [ ]:
len(results["objects"])

47

In [ ]:
objects = [x for x in results["objects"]]

In [ ]:
objects

In [ ]:
objects[0].keys()

dict_keys(['class', 'score', 'box'])

In [ ]:
objects[0]

In [ ]:
classes = [x["class"] for x in objects]

In [ ]:
{item: classes.count(item) for item in set(classes)}

### 3. 이미지에 박스 그리기

In [ ]:
import cv2
from google.colab.patches import cv2_imshow
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image, ImageDraw, ImageFont

In [ ]:
image = cv2.imread(IMAGE_FILE_PATH)

In [ ]:
obj = objects[0]

In [ ]:
obj

In [ ]:
obj_class = obj["class"]
score = obj["score"]
x1, y1, x2, y2 = obj["box"]

In [ ]:
image = cv2.rectangle(image, (int(x1), int(y1)), (int(x2), int(y2)), (255, 0 , 0), 5)

In [ ]:
# PIL 이미지로 변환
pil_image = Image.fromarray(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))

# 한글 폰트 설정 (폰트 경로 필요)
font_path = "malgun.ttf"  # Windows의 경우 'malgun.ttf', Mac은 다른 한글 폰트 사용
font = ImageFont.truetype(font_path, 32)  # 폰트 크기 32 설정

# PIL로 그림 그리기
draw = ImageDraw.Draw(pil_image)
draw.text((int(x1), int(y1) - 50), f"{obj_class} {score:.2f}", font=font, fill=(255, 0 , 0))

# 다시 OpenCV 이미지로 변환
image_with_text = cv2.cvtColor(np.array(pil_image), cv2.COLOR_RGB2BGR)

In [ ]:
cv2_imshow(image_with_text)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [ ]:
image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)  # BGR -> RGB 변환

# 이미지 크기 확인
height, width, _ = image.shape

# 원본 크기로 출력 (픽셀 크기를 인치로 변환: width / DPI, height / DPI)
dpi = 100  # DPI 설정
figsize = (width / dpi, height / dpi)

plt.figure(figsize=figsize)
plt.imshow(image_rgb)
plt.axis('off')  # 축 제거
plt.title('Original Size Image')
plt.show()

### 4. 모든 객체 표현

In [ ]:
import random

In [ ]:
def generate_random_color():
    """
    Generate a random RGB color.

    Returns:
        tuple: A tuple containing RGB values (R, G, B) with each value in the range 0-255.
    """
    r = random.randint(0, 255)  # Red
    g = random.randint(0, 255)  # Green
    b = random.randint(0, 255)  # Blue
    return (r, g, b)

In [ ]:
# 한글 폰트 설정 (폰트 경로 필요)
font_path = "malgun.ttf"  # Windows의 경우 'malgun.ttf', Mac은 다른 한글 폰트 사용
font = ImageFont.truetype(font_path, 32)  # 폰트 크기 32 설정
class_color = {}

In [ ]:
def draw_inference_results(img: np.ndarray, obj_class: str, score: float, box: list):
  if obj_class not in class_color:
      color = generate_random_color()
      class_color[obj_class] = color
  else:
      color = class_color[obj_class]

  x1, y1, x2, y2 = box

  img = cv2.rectangle(img, (int(x1), int(y1)), (int(x2), int(y2)), color, 5)

  # PIL 이미지로 변환
  pil_image = Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))

  # PIL로 그림 그리기
  draw = ImageDraw.Draw(pil_image)
  draw.text((int(x1), int(y1) - 50), f"{obj_class} {score:.2f}", font=font, fill=color)  # 흰색 텍스트

  # 다시 OpenCV 이미지로 변환
  img = cv2.cvtColor(np.array(pil_image), cv2.COLOR_RGB2BGR)

  return img

In [ ]:
image = cv2.imread(IMAGE_FILE_PATH)

In [ ]:
for obj in objects:
  image = draw_inference_results(img=image, obj_class=obj["class"], score=obj["score"], box=obj["box"])

In [ ]:
cv2_imshow(image)
cv2.waitKey(0)
cv2.destroyAllWindows()

### 5. 모든 이미지에 추론 적용 후 저장

In [ ]:
import os

In [ ]:
# 프레임을 저장할 디렉토리 생성
output_dir = "inference_results"
os.makedirs(output_dir, exist_ok=True)

In [ ]:
input_dir = "images"

In [ ]:
# 플랫폼에서 4메가 이상의 파일은 처리하지 못함
max_size_mb = 4

# 파일 용량 필터링
images = [
    file for file in os.listdir(input_dir)
    if os.path.isfile(os.path.join(input_dir, file)) and
    os.path.getsize(os.path.join(input_dir, file)) <= max_size_mb * 1024 * 1024
]

# 파일 확장자 필터링
images = [x for x in images if x.endswith(('.jpeg', '.jpg'))]

In [ ]:
for img in images:
  IMAGE_FILE_PATH = f"images/{img}"
  image = open(IMAGE_FILE_PATH, "rb").read()

  response = requests.post(
      url=URL,
      auth=HTTPBasicAuth("cheil-jedang", ACCESS_KEY),
      headers={"Content-Type": "image/jpeg"},
      data=image,
  )

  results = response.json()

  objects = [x for x in results["objects"]]

  image = cv2.imread(f"images/{img}")

  for obj in objects:
    image = draw_inference_results(img=image, obj_class=obj["class"], score=obj["score"], box=obj["box"])

  cv2.imwrite(f"inference_results/{img}", image)

  print(f"{img} 처리 완료")

### 6. 폴더 압축

In [ ]:
import shutil

# 압축할 폴더 경로와 압축 파일 저장 경로 설정
folder_to_compress = "inference_results"  # 압축할 폴더
output_archive = "results"  # 압축 파일 경로 (확장자 없이)

# 폴더 압축
shutil.make_archive(output_archive, 'zip', folder_to_compress)

print(f"폴더가 압축되었습니다: {output_archive}.zip")

폴더가 압축되었습니다: results.zip
